
# C7-cnn-transfer — Session 2: Reading ResNet-50

*One class session, roughly 85 minutes. Prerequisites: Session 1
(convolution, feature maps, the output-size formula, receptive
fields) and C6 (modules, `named_children`/`named_parameters`,
parameter counting, `state_dict`).*

**This session:** stop building toy networks and read a real one.
`torchvision` ships **ResNet-50**, a 50-layer convolutional network
with pretrained ImageNet weights — the exact model the real exam hands
you — and every C6 inspection tool works on it unchanged.
The agenda: load it *reproducibly* (three habits: explicit weights,
`eval()`, `inference_mode()`); read its top-level anatomy with
`named_children`; trace every stage's output shape by hand and verify
in code; open up its repeating unit, the **bottleneck block**; and
count its parameters by hand at every scale, `numel` only as the
check.
No training, no gradients — a pretrained network is a finished
artifact, and this session is the art of reading one.


In [1]:

# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception, stated once per notebook): pretrained
# resnet50 is a float32 artifact, so this notebook does NOT set the float64
# default. Inputs are cast .to(torch.float32) at the model boundary; float
# comparisons state atol=1e-6 / rtol=1e-5.
SEED = 20260804

print("torch", torch.__version__)


torch 2.13.0+cpu



## 1. Loading a Pretrained Model, Reproducibly

`torchvision.models` is a **model zoo**: architectures plus trained
weights, downloadable by name.
Three habits make a load reproducible, and all three are course
convention from here on:

1. **Name the weights explicitly.**
   `weights=ResNet50_Weights.IMAGENET1K_V1` pins the exact artifact —
   the same tensors every time, on every machine.
   (The legacy `pretrained=True` is deprecated precisely because it
   leaves the choice to the library.)
2. **`model.eval()` immediately after loading.**
   Some layers behave differently in training mode; ResNet's
   **BatchNorm** layers, in train mode, normalize each batch by *its
   own* statistics — so the same input gives different outputs
   depending on what it is batched with, and each forward pass
   *mutates* the layer's running buffers.
   `eval()` switches them to their stored statistics: deterministic,
   and side-effect-free.
3. **Every forward pass inside `torch.inference_mode()`.**
   A pretrained model's parameters arrive with
   `requires_grad=True` (intent: further training), so a bare forward
   silently builds an autograd graph we will never use.
   `inference_mode()` turns all of that machinery off — the honest
   register for a course that only ever runs forward.

One more pin, the mirror image of C6's dtype discipline: the
downloaded weights are **float32** — that *is* the artifact — so this
notebook leaves torch's float32 default in place rather than forcing
float64, casts inputs to `torch.float32` at the model boundary, and
states the honest float32 tolerances (`atol=1e-6`, `rtol=1e-5`)
whenever values are compared.


In [2]:

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()                                   # BatchNorm -> stored statistics

w_dtype = next(model.parameters()).dtype
print("weights dtype:", w_dtype, " training mode:", model.training)
assert w_dtype == torch.float32

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)   # cast at the model boundary

with torch.inference_mode():
    logits = model(x)
    again = model(x)

print("output shape:", tuple(logits.shape))
print("two runs bit-identical:", bool((logits == again).all()))
print("logits value range: [%.3f, %.3f]" % (float(logits.min()), float(logits.max())))


weights dtype: torch.float32  training mode: False
output shape: (2, 1000)
two runs bit-identical: True
logits value range: [-4.921, 8.746]



Read the printout as three verified claims: the artifact is float32
and in eval mode; the output for a batch of two 224×224 RGB inputs is
`(2, 1000)` — one score per ImageNet class; and the forward pass is
**bit-identical** across runs — `eval()` plus fixed weights leaves no
randomness anywhere.

*Why then state tolerances at all?*
Because float32 carries only ~7 significant digits: the moment the
*same quantity* is computed along two different routes (a different
summation order, a value recorded in a notebook and recomputed later),
the last digits wobble.
The demonstration below computes the same logits through a float64
copy of the model — the arithmetic differs at the seventh digit or so,
and on logits of size $O(10)$ (the printed range) that lands near
$10^{-6}$ absolute:


In [3]:

model64 = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval().to(torch.float64)
with torch.inference_mode():
    logits64 = model64(x.to(torch.float64))

gap = (logits.to(torch.float64) - logits64).abs()
print("max abs gap float32-vs-float64 route: %.2e" % float(gap.max()))
print("allclose at atol=1e-6/rtol=1e-5:",
      torch.allclose(logits.to(torch.float64), logits64, atol=1e-6, rtol=1e-5))
del model64, logits64


max abs gap float32-vs-float64 route: 2.56e-06
allclose at atol=1e-6/rtol=1e-5: False



The measured gap is ≈ `2.6e-06` — real, harmless, and *just outside*
`atol=1e-6` where a logit sits near zero (relative tolerance cannot
rescue an anchor whose true value is tiny).
Hence the course's two-part rule for this unit:

- anchors for ResNet-derived values are always **recorded from the
  float32 pipeline itself** (never from a float64 rerun), and checked
  with `atol=1e-6`/`rtol=1e-5` — same-route recomputation sits far
  inside that;
- a *cross*-precision comparison like the one above is a diagnostic,
  not an anchor check, and when a problem ever needs one it widens
  the tolerance explicitly, with a comment.

### Checkpoint 1

1. Name the three reproducibility habits and, for each, the failure
   it prevents (one clause each).
2. A teammate loads with `pretrained=True`, skips `eval()`, and runs
   the same image twice inside one batch versus alongside 63 random
   images.
   Which of their comparisons can disagree, and through which layer's
   mechanism?
3. Why is a *repeat* of the same float32 forward pass bit-identical,
   while the float64-route comparison shows a $10^{-6}$-scale gap —
   what differs between the two situations?



## 2. Top-Level Anatomy via `named_children`

C6 read hand-built modules with `named_children()`; ResNet-50 answers
the same call:


In [4]:

for name, child in model.named_children():
    print(f"{name:8s} {type(child).__name__}")


conv1    Conv2d
bn1      BatchNorm2d
relu     ReLU
maxpool  MaxPool2d
layer1   Sequential
layer2   Sequential
layer3   Sequential
layer4   Sequential
avgpool  AdaptiveAvgPool2d
fc       Linear



Ten children, in forward order, and they group naturally:

| group | children | role |
|---|---|---|
| **stem** | `conv1` (7×7, stride 2), `bn1`, `relu`, `maxpool` (3×3, stride 2) | compress the raw image fast: 224 → 56 per side |
| **stages** | `layer1` … `layer4` | the convolutional body — each an `nn.Sequential` of repeated **bottleneck blocks** |
| **head** | `avgpool` (adaptive, to 1×1), `fc` (`Linear(2048, 1000)`) | collapse the final maps and score 1000 classes |

The stages are where depth lives.
Each `layerN` is an `nn.Sequential`, so `len()` counts its blocks:


In [5]:

blocks = [len(getattr(model, f"layer{i}")) for i in (1, 2, 3, 4)]
print("blocks per stage:", blocks)

n_convs_body = 3 * sum(blocks)     # each bottleneck block holds exactly 3 convs
print("convs in the body:", n_convs_body,
      " + stem conv + fc =", n_convs_body + 2)


blocks per stage: [3, 4, 6, 3]
convs in the body: 48  + stem conv + fc = 50



`[3, 4, 6, 3]` blocks, each holding 3 convolutions (Section 4), gives
$3 \times 16 = 48$; add `conv1` and the final `fc` and you get **50**
weighted layers — the "50" in ResNet-50.
The architecture in one sentence: *a stem, then 3+4+6+3 bottleneck
blocks in four stages, then pool-and-score.*

### Checkpoint 2

1. From the table, which children own **no** parameters at all?
   (Three of the ten — say why for each.)
2. `model.layer3[5]` — what does this expression address, and why
   does `model.layer3[6]` raise?
3. Predict `type(model.layer2).__name__` and what `len(model.layer2)`
   returns, then check both in a scratch cell.



## 3. Stage Output Shapes: Trace by Hand, Verify in Code

Session 1's output-size formula
$n_\text{out} = \lfloor (n + 2p - K)/s \rfloor + 1$ plus two ResNet
facts lets you trace the whole network on paper:

- the **stem**: `conv1` is $7\times7$, stride 2, padding 3
  ($224 \to 112$); `maxpool` is $3\times3$, stride 2, padding 1
  ($112 \to 56$);
- each of `layer2`, `layer3`, `layer4` **halves the grid once** (its
  first block carries a stride-2 conv); `layer1` keeps the grid.

Channel counts per stage are architecture constants:
$256, 512, 1024, 2048$ out of `layer1..4`.
The hand trace for a `(B, 3, 224, 224)` input:

| after | shape | why |
|---|---|---|
| `conv1` | $(B, 64, 112, 112)$ | $\lfloor(224 + 6 - 7)/2\rfloor + 1 = 112$ |
| `maxpool` | $(B, 64, 56, 56)$ | $\lfloor(112 + 2 - 3)/2\rfloor + 1 = 56$ |
| `layer1` | $(B, 256, 56, 56)$ | stride 1 throughout |
| `layer2` | $(B, 512, 28, 28)$ | halves once |
| `layer3` | $(B, 1024, 14, 14)$ | halves once |
| `layer4` | $(B, 2048, 7, 7)$ | halves once |
| `avgpool` | $(B, 2048, 1, 1)$ | adaptive average to $1\times1$ |
| `fc` | $(B, 1000)$ | after flattening to $(B, 2048)$ |

Verification is a loop over the children — with one wrinkle: the
module's own `forward` flattens between `avgpool` and `fc`, and a
child-by-child replay must do the same by hand:


In [6]:

cur = x                                   # (2, 3, 224, 224), float32
with torch.inference_mode():
    for name, child in model.named_children():
        if name == "fc":
            cur = torch.flatten(cur, 1)   # what model.forward does between avgpool and fc
        cur = child(cur)
        print(f"after {name:8s} {tuple(cur.shape)}")


after conv1    (2, 64, 112, 112)
after bn1      (2, 64, 112, 112)
after relu     (2, 64, 112, 112)
after maxpool  (2, 64, 56, 56)
after layer1   (2, 256, 56, 56)
after layer2   (2, 512, 28, 28)
after layer3   (2, 1024, 14, 14)
after layer4   (2, 2048, 7, 7)
after avgpool  (2, 2048, 1, 1)
after fc       (2, 1000)



Every row matches the hand table.
Two readings to internalize:

- **The grid shrinks $224 \to 7$ while channels grow $3 \to 2048$** —
  Session 1 §8's shape signature of the feature hierarchy, now read
  off a real network.
- `avgpool` is *adaptive*: it averages whatever grid arrives down to
  $1\times1$, which is why other input sizes work too — only the
  spatial dims change along the way, never the channel counts.
  A `(1, 3, 160, 160)` input runs
  $80 \to 40 \to 40 \to 20 \to 10 \to 5 \to 1$ through the same
  layers (trace it with the formula; the halving steps are exact
  because stride-2 layers here have the padding that makes
  $\lfloor n/2 \rfloor$ land clean).

### Checkpoint 3

1. Hand-trace a `(1, 3, 160, 160)` input through all ten children
   (spatial size after each of stem, layer1–4, avgpool) and verify
   with the loop above.
2. Which single word in "adaptive average pool" is doing the work
   that makes the fc layer's input size independent of the image
   size?
   What breaks first if you replace `avgpool` with a fixed
   `nn.AvgPool2d(7)` and feed $160\times160$?
3. Between which two children does `torch.flatten(cur, 1)` sit, and
   what shape does it turn $(B, 2048, 1, 1)$ into?



## 4. Inside a Stage: the Bottleneck Block

Print one block — the third block of `layer2`, an interior block
chosen so nothing special is going on:


In [7]:

print(model.layer2[2])


Bottleneck(
  (conv1): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(128, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn3): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)



Read the printout top to bottom — three convolutions, each followed by
a BatchNorm:

1. `conv1`: $1\times1$, $512 \to 128$ — the **reduce**: a $1\times1$
   conv mixes *channels at a single position* (it is exactly a dense
   layer applied at every pixel), here compressing 512 channels to
   128;
2. `conv2`: $3\times3$, $128 \to 128$, padding 1 — the only *spatial*
   look, done cheaply in the thin 128-channel space;
3. `conv3`: $1\times1$, $128 \to 512$ — the **expand**, back to 512.

Thin in the middle, wide at the ends: the **bottleneck** shape.
The block's output is not just `conv3`'s output — a **skip
connection** adds the block's *input* back before the final ReLU
(that is the "residual" in ResNet; the printout doesn't show it
because it lives in the block's `forward`, not in a submodule).
So the block computes `relu(F(x) + x)`: each block *adjusts* the
running representation rather than replacing it.

Constants worth pinning: every bottleneck's channel trio is
$(4m \to m \to m \to 4m)$ for stage width $m$ — the **expansion
factor** is 4; the stage widths $m$ are $64, 128, 256, 512$ for
`layer1..4` (hence outputs $256, 512, 1024, 2048$).

**Stage-opening blocks.**
Block 0 of each stage must reconcile the skip with a shape change
(new channel count, and — except `layer1` — a halved grid).
It carries a `downsample` branch: a $1\times1$ conv (stride 2 where
the grid halves) plus BatchNorm that maps the *input* to the new
shape so the addition still lines up:


In [8]:

opener = model.layer2[0]
print("interior block has downsample:", hasattr(model.layer2[2], "downsample")
      and model.layer2[2].downsample is not None)
print("stage-opening block downsample:\n", opener.downsample)
print("conv2 stride in opener:", opener.conv2.stride,
      " in interior block:", model.layer2[2].conv2.stride)


interior block has downsample: False
stage-opening block downsample:
 Sequential(
  (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
  (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)
conv2 stride in opener: (2, 2)  in interior block: (1, 1)



The interior block's `downsample` is `None`; the opener carries the
$1\times1$ stride-2 conv + BN pair, and its `conv2` is the stride-2
member (`(2, 2)` vs `(1, 1)`) — the halving read off Section 3's
shape table happens exactly here.

### Checkpoint 4

1. For `layer4`'s interior blocks: the three conv shapes (in/out
   channels, kernel) from the constants alone — then verify one with
   `print(model.layer4[1])`.
2. Why is a $1\times1$ convolution "a dense layer at every pixel" —
   what does its weight tensor `(out, in, 1, 1)` contract over, and
   what does it leave alone?
3. Which blocks of the whole network carry a `downsample`, and which
   of those also halve the grid? (One does not — which, and why?)



## 5. Parameter Arithmetic: Counting a Real Block by Hand

C6's counting register, one level up.
The atoms:

- **Convolution**: a weight of shape
  $(\text{out}, \text{in}, K, K)$ holds
  $\text{out} \times \text{in} \times K \times K$ numbers — and
  *every conv in ResNet has* `bias=False` (the printout said so), so
  that product is the whole count.
- **BatchNorm**: $2C$ learnable parameters for $C$ channels (a scale
  and a shift per channel) — stated here as a counted-but-not-derived
  fact; its running statistics are *buffers*, not parameters, and
  never enter the count.

Warm-up, the stem conv: $64 \times 3 \times 7 \times 7 = 9{,}408$.

**Worked count — all of `layer2[2]`** ($512 \to 128 \to 128 \to 512$):

| piece | arithmetic | count |
|---|---|---|
| `conv1` $1\times1$ | $128 \times 512 \times 1 \times 1$ | $65{,}536$ |
| `conv2` $3\times3$ | $128 \times 128 \times 3 \times 3$ | $147{,}456$ |
| `conv3` $1\times1$ | $512 \times 128 \times 1 \times 1$ | $65{,}536$ |
| convs total | | $278{,}528$ |
| `bn1`, `bn2`, `bn3` | $2(128 + 128 + 512)$ | $1{,}536$ |
| **block total** | | $\mathbf{280{,}064}$ |

The exam's **count-without-`numel` register** applies verbatim: when a
problem grades the arithmetic, `numel`, `sum(p.numel() ...)`,
`torchsummary`, and reading sizes off `state_dict` are banned (zero
points) — the multiplication *is* the answer.
`numel` then reappears as the *check*:


In [9]:

hand = 128 * 512 + 128 * 128 * 3 * 3 + 512 * 128        # convs
hand_bn = 2 * (128 + 128 + 512)                          # +BN pairs
total = sum(p.numel() for p in model.layer2[2].parameters())
print("hand convs:", hand, " +BN:", hand + hand_bn, " numel check:", total)
assert hand + hand_bn == total


hand convs: 278528  +BN: 280064  numel check: 280064



`280,064` both ways.

---

**Worked exam-style example (multiple choice, numeric normal form).**

*Let $N$ be the number of parameters in the three convolutions
(weights only — ResNet convs have no bias; exclude BatchNorm) of an
interior bottleneck block of `layer1`, i.e. channels
$256 \to 64 \to 64 \to 256$.
$N$ can be written uniquely as $N = 2^a \cdot b$ with $a \ge 0$ an
integer and $b$ odd. What is $a + b$?*

A. 21  B. 25  C. 29  D. 33  E. 69,632

*Solution.*
$N = 64\cdot256 \;+\; 64\cdot64\cdot9 \;+\; 256\cdot64
= 16{,}384 + 36{,}864 + 16{,}384 = 69{,}632$.
Decode: $69{,}632 = 2^{12} \cdot 17$ (halve twelve times: $69{,}632
\to 34{,}816 \to \dots \to 17$), so $a + b = 12 + 17 = 29$:
**answer C.**
Traps: dropping one of the $1\times1$s gives $53{,}248 = 2^{12}\cdot13
\mapsto 25$ (B); forgetting the $\times9$ on the middle conv gives
$36{,}864 = 2^{12}\cdot9 \mapsto 21$ (A); including BN's $2(64+64+256)
= 768$ gives $70{,}400 = 2^7\cdot550$ — not odd — redone correctly
$2^8 \cdot 275 \mapsto 283$, not offered, so the sanity check "my
decode must be a choice" catches it; answering $N$ itself (E) ignores
the normal form.

---

### Checkpoint 5

1. Hand-count the three convs of a `layer3` interior block
   ($1024 \to 256 \to 256 \to 1024$) and its three BNs; check with
   `numel` in a scratch cell.
2. Why do the two $1\times1$ convs of any bottleneck contribute
   *equally*? Does that survive in the stage-opening blocks?
3. In the worked MC, the fraction of $N$ contributed by the
   $3\times3$ conv is $36{,}864/69{,}632$.
   Reduce it to lowest terms — and explain why the same fraction
   appears in *every* interior bottleneck of *every* stage.



## 6. Counting at Model Scale

The same arithmetic, aggregated.
Per top-level child (hand-checkable piece by piece, `numel` doing the
bookkeeping):


In [10]:

total = 0
for name, child in model.named_children():
    n = sum(p.numel() for p in child.parameters())
    total += n
    print(f"{name:8s} {n:>12,}")
print(f"{'total':8s} {total:>12,}")


conv1           9,408
bn1               128
relu                0
maxpool             0
layer1        215,808
layer2      1,219,584
layer3      7,098,368
layer4     14,964,736
avgpool             0
fc          2,049,000
total      25,557,032



Read the column like an auditor:

- `relu`, `maxpool`, `avgpool`: **0** — no learnable numbers, exactly
  as Checkpoint 2 predicted.
- The body dominates, and *later stages dwarf earlier ones*:
  `layer4` alone holds $14{,}964{,}736$ of the
  $25{,}557{,}032$ total — about 59%.
  Same block structure, wider channels, and counts grow with the
  *square* of width (all three conv terms are products of two channel
  counts).
- `fc` is a single dense layer yet holds $2{,}049{,}000$
  parameters: $2048 \times 1000 + 1000$ — C6's
  $\text{out}\times(\text{in}+1)$, at scale.
  Handy closed form: a $2048 \to k$ head holds $2048k + k = 2049k$
  parameters — remember it, Session 3 sizes fresh heads with it.
- The grand total $25{,}557{,}032$ is the published ResNet-50
  figure — your audit reproduces it.

### Checkpoint 6

1. Verify the 59% claim from the printed column, and compute
   `layer1`'s share to one decimal.
   Why *must* later stages dominate, given the $m, 4m$ channel
   structure?
2. Using the $2049k$ closed form: how many parameters would `fc` hold
   with 10 output classes? With 1 class? Check the 1000-class value
   against the printed column.
3. `bn1` prints 128 — reconcile: how many channels does `conv1`
   output, and what is BN's per-channel cost?
   How many numbers does `bn1` *store* that are **not** parameters
   (name them)?



## 7. Common Pitfalls II

**Pitfall 1 — forgetting `eval()`: same input, different answers.**
BatchNorm in train mode normalizes by the *current batch*, so the
company an input keeps changes its output — and every forward pass
also shifts the running buffers.
Demonstrate on a throwaway copy of one block (never mutate the real
model):


In [11]:

import copy

blk = copy.deepcopy(model.layer2[2])       # sacrificial copy
torch.manual_seed(SEED)
u = torch.randn(4, 512, 14, 14).to(torch.float32)

blk.eval()
with torch.inference_mode():
    out_eval_a = blk(u)
    out_eval_b = blk(u[:2])                # same first two items, smaller batch
print("eval  mode, item 0 stable across batch composition:",
      bool(torch.equal(out_eval_a[:2], out_eval_b)))

blk.train()                                # what forgetting eval() means
buf_before = blk.bn2.running_mean.clone()
with torch.inference_mode():
    out_tr_a = blk(u)
    out_tr_b = blk(u[:2])
print("train mode, item 0 stable across batch composition:",
      bool(torch.equal(out_tr_a[:2], out_tr_b)))
print("train mode mutated running_mean:",
      bool(not torch.equal(buf_before, blk.bn2.running_mean)))
del blk


eval  mode, item 0 stable across batch composition: True
train mode, item 0 stable across batch composition: False
train mode mutated running_mean: True



`True / False / True`: in train mode the same two inputs answer
differently depending on batch mates, and the block's buffers moved
just from being *run*.
`eval()` is the single switch that makes forward passes deterministic
— which is why the course calls it in the same breath as loading.

**Pitfall 2 — course-reflex float64 input at the float32 boundary.**
C6's habit of casting *up* to float64 is exactly wrong here:


In [12]:

try:
    with torch.inference_mode():
        model.conv1(x.to(torch.float64))
except RuntimeError as e:
    print("RuntimeError:", str(e)[:80], "...")
print("the cast goes DOWN at this boundary:",
      tuple(model.conv1(x.to(torch.float32)).shape))


RuntimeError: expected scalar type Double but found Float ...
the cast goes DOWN at this boundary: (2, 64, 112, 112)



The artifact's dtype wins: cast the *input* to `torch.float32` at the
model boundary, every time.
(The message is famously terse — `expected scalar type Double but
found Float` reports that a `Double` and a `Float` tensor met without
saying which was yours; the boundary cast resolves it either way.)

**Pitfall 3 — counting buffers as parameters.**
`state_dict` stores everything needed to reconstruct the model —
parameters *and* buffers — so sizing a model by its `state_dict` keys
overcounts:


In [13]:

bn = model.bn1
n_params = sum(p.numel() for p in bn.parameters())
n_state = sum(v.numel() for v in bn.state_dict().values())
print("bn1 parameters:", n_params, " state_dict numbers:", n_state)
print("state_dict keys:", list(bn.state_dict().keys()))


bn1 parameters: 128  state_dict numbers: 257
state_dict keys: ['weight', 'bias', 'running_mean', 'running_var', 'num_batches_tracked']



`128` parameters but `257` stored numbers: `running_mean` (64),
`running_var` (64), and the scalar `num_batches_tracked` ride along in
the state dict.
Parameter questions ask `parameters()`; checkpoint-size questions ask
`state_dict` — keep the registers apart.

**Pitfall 4 — `children()` is a generator.**
It yields each child once; a second pass over the same generator
object finds nothing.
List it once, slice the list many times (Session 3 does exactly this):


In [14]:

gen = model.children()
first_pass = [type(c).__name__ for c in gen]
second_pass = [type(c).__name__ for c in gen]
print("first pass:", len(first_pass), " second pass:", len(second_pass))
ch = list(model.children())                # the idiom: list once, reuse
print("listed once, reusable:", len(ch), "and again:", len(list(ch)))


first pass: 10  second pass: 0
listed once, reusable: 10 and again: 10



`10` then `0` — the exhausted-generator trap; the `list(...)` idiom
never surprises.

### Checkpoint 7

1. A notebook reports that the "same" image classified alone and in a
   batch of 32 gets different top-1 scores, but only on a colleague's
   machine.
   Which pitfall is the prime suspect, what one line fixes it, and
   why can the bug hide on machines that only ever run batch size 1?
2. Estimate (then check by aggregating `state_dict` values minus
   parameters over the whole model): are ResNet-50's non-parameter
   stored numbers closer to 50 thousand or 5 million?
   Which layers contribute them?
3. Write the one-line failure: what does
   `slices = model.children(); a = nn.Sequential(*slices);
   b = nn.Sequential(*slices)` leave in `b`, and how does the
   list-once idiom prevent it?


## 8. Tensor Shape Tracing: Channels and Grids

A convolution changes two different parts of \((B,C,H,W)\), by two different rules.

- **Channels:** the layer declaration gives them directly. `Conv2d(C_in, C_out, ...)` replaces \(C_{in}\) by \(C_{out}\); kernel, stride, and padding do not decide channel count.
- **Spatial grid:** for each axis separately,
  \[
  n_{out}=\left\lfloor\frac{n_{in}+2p-k}{s}\right\rfloor+1
  \]
  at dilation 1. Tuple arguments mean the height and width arithmetic may differ.
- **Downsampling:** stride greater than 1 skips window origins and usually shrinks the grid. Padding can preserve a stride-1 grid, but it does not cancel a larger stride.

Trace one layer at a time and carry the full tuple forward. For input \((2,17,179,193)\), let `conv_a` map \(17\to37\) with kernel \((7,5)\), stride \((3,2)\), padding \((2,1)\); then let `conv_b` map \(37\to53\) with kernel 3, stride 2, padding 1. The hand trace is

\[
(2,17,179,193)\to(2,37,59,96)\to(2,53,30,48).
\]

The first height is \(\lfloor(179+4-7)/3\rfloor+1=59\); its width is \((193+2-5)/2+1=96\). The next layer starts from **those** numbers, not the original grid. A residual addition adds one more contract: the skip path and main path must finish with identical channel, height, and width values. If either channels or spatial downsampling changes, the skip needs a matching projection.


In [15]:
shape_demo = nn.Sequential(
    nn.Conv2d(17, 37, kernel_size=(7, 5), stride=(3, 2), padding=(2, 1)),
    nn.Conv2d(37, 53, kernel_size=3, stride=2, padding=1),
)
shape_x = torch.zeros(2, 17, 179, 193)
shape_observed = []
with torch.inference_mode():
    for layer in shape_demo:
        shape_x = layer(shape_x)
        shape_observed.append(tuple(shape_x.shape))
print(shape_observed)


[(2, 37, 59, 96), (2, 53, 30, 48)]


The printout agrees with the hand trace: `[(2, 37, 59, 96), (2, 53, 30, 48)]`. Code is a check after the arithmetic, never a substitute for it.

### Checkpoint 8

1. Starting from \((4,19,257,263)\), a main path applies \(19\to43\), kernel 5, stride 2, padding 2; then \(43\to61\), kernel 3, stride \((2,1)\), padding 1. Give both output tuples.
2. A skip projection goes directly from the checkpoint input to the main path's final output with kernel 1 and padding 0. What output-channel count and stride tuple make the addition legal?
3. A trace changes \(C\) using the stride and changes \(H,W\) using `out_channels`. Diagnose both mistakes in one sentence.



## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Explicit `weights=` — pins the artifact (no silent weight-version
   drift); `eval()` — stops BatchNorm from using batch statistics and
   mutating buffers (nondeterminism + side effects);
   `inference_mode()` — stops autograd from building graphs no one
   will use (silent memory/compute waste).
2. Alone-vs-in-a-crowd comparisons can disagree — in train mode every
   BatchNorm normalizes by the current batch's mean/variance, so batch
   mates leak into each item's output.
   (The `pretrained=True` habit costs reproducibility across library
   versions, not within one run.)
3. A repeat run executes the identical float32 operations in the
   identical order — bit-identical is guaranteed.
   The float64 route performs *different arithmetic* (higher
   precision, different rounding), so it lands within float32's ~7
   significant digits of the same answer, not on it.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `relu` (a fixed function), `maxpool` (a fixed reduction), and
   `avgpool` (a fixed average) — no learnable numbers anywhere in the
   three.
2. The sixth (last) bottleneck block of stage 3 — `layer3` is an
   `nn.Sequential` of 6 blocks, indexed 0–5; index 6 is out of range.
3. `Sequential`, and `len(model.layer2)` is `4`.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Stem: $\lfloor(160+6-7)/2\rfloor+1 = 80$, then maxpool
   $\lfloor(80+2-3)/2\rfloor+1 = 40$; `layer1` 40; `layer2` 20;
   `layer3` 10; `layer4` 5; `avgpool` 1.
   Shapes: $(1,64,80,80) \to (1,64,40,40) \to (1,256,40,40) \to
   (1,512,20,20) \to (1,1024,10,10) \to (1,2048,5,5) \to
   (1,2048,1,1) \to (1,1000)$.
2. *Adaptive*: the layer computes whatever kernel size reduces the
   incoming grid to the requested $1\times1$.
   A fixed `AvgPool2d(7)` on the $5\times5$ `layer4` output has no
   room for its $7\times7$ window — the forward crashes there (and
   with other sizes could instead silently emit a grid larger than
   $1\times1$ and break `fc`'s input size).
3. Between `avgpool` and `fc`; it turns $(B, 2048, 1, 1)$ into
   $(B, 2048)$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `conv1`: $(512, 2048, 1, 1)$ i.e. $2048 \to 512$ $1\times1$;
   `conv2`: $(512, 512, 3, 3)$; `conv3`: $(2048, 512, 1, 1)$ —
   stage width $m = 512$, ends $4m = 2048$.
2. Its weight contracts over input channels only (the $1\times1$
   spatial extent means no neighbors are read): at each pixel, the
   output channel vector is a matrix product of the input channel
   vector — a dense layer applied position-wise.
   It leaves the spatial arrangement alone.
3. Block 0 of every stage (`layer1[0]`, `layer2[0]`, `layer3[0]`,
   `layer4[0]`).
   All except `layer1[0]` also halve the grid; `layer1[0]`'s
   downsample is stride 1 — it exists only to fix the *channel*
   mismatch ($64 \to 256$) for the skip addition, the grid having
   already been halved by the stem's maxpool.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Convs: $256\cdot1024 + 256\cdot256\cdot9 + 1024\cdot256 =
   262{,}144 + 589{,}824 + 262{,}144 = 1{,}114{,}112$;
   BNs: $2(256+256+1024) = 3{,}072$; block total $1{,}117{,}184$ —
   `sum(p.numel() for p in model.layer3[1].parameters())` agrees.
2. Reduce is $m \times 4m$ numbers, expand is $4m \times m$ — equal
   by symmetry of the product.
   No: an opener's reduce reads the *previous* stage's width $2m$
   (e.g. `layer2[0].conv1` is $256 \to 128$), while its expand still
   writes $4m$, so the two differ there (and the downsample conv adds
   its own $4m \times 2m$).
3. $36{,}864/69{,}632 = 9/17$ (divide by $2^{12} = 4096$).
   In any interior bottleneck the three convs hold
   $m\cdot4m,\; 9m^2,\; 4m\cdot m$ — i.e. $4m^2, 9m^2, 4m^2$ — so
   the $3\times3$'s share is $9m^2/17m^2 = 9/17$ regardless of $m$:
   the ratio is a property of the block *shape*, not its width.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $14{,}964{,}736 / 25{,}557{,}032 \approx 0.586 \to 59\%$;
   `layer1`: $215{,}808 / 25{,}557{,}032 \approx 0.8\%$.
   Every conv count is a product of two channel widths, and widths
   double per stage — counts scale like $m^2$, so each stage runs
   roughly $4\times$ its predecessor (block counts modulate this).
2. $2049 \cdot 10 = 20{,}490$; $2049 \cdot 1 = 2{,}049$;
   $2049 \cdot 1000 = 2{,}049{,}000$ — the printed `fc` row.
3. `conv1` outputs 64 channels; BN costs $2$ per channel $= 128$
   parameters.
   Non-parameters: `running_mean` (64), `running_var` (64), and
   `num_batches_tracked` (1) — 129 stored numbers that never enter a
   parameter count.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pitfall 1 (missing `eval()`); fix: `model.eval()` after loading.
   At batch size 1 the batch statistics *are* that image's own
   statistics every time — consistently wrong but consistent, so the
   discrepancy only surfaces when batch composition varies.
2. ≈ 50 thousand: every BatchNorm stores $2C + 1$ buffer numbers, and
   summing `state_dict` values minus parameters over the model gives
   53,173 exactly (the check:
   `sum(v.numel() for v in model.state_dict().values()) -
   sum(p.numel() for p in model.parameters())`).
   Only the BN layers contribute — convs and `fc` store parameters
   only.
3. `a` consumes the generator; `b` receives *no* children — an empty
   `Sequential` that silently maps any input to itself.
   `ch = list(model.children())` materializes once; every later
   `nn.Sequential(*ch[:k])` reads the same list.

</details>


<details><summary><b>Checkpoint 8</b></summary>

1. First \((4,43,129,132)\): each axis is \(\lfloor(n+4-5)/2\rfloor+1\). Then \((4,61,65,132)\): height is downsampled again while stride 1 preserves width.
2. It must emit 61 channels with stride \((4,2)\). A kernel-1, padding-0 projection gives height \(\lfloor(257-1)/4\rfloor+1=65\) and width \(\lfloor(263-1)/2\rfloor+1=132\).
3. The mistakes swap independent contracts: `out_channels` determines \(C\), while kernel/stride/padding arithmetic determines \(H,W\).

</details>
